# Module 0: Setup
> ~15 min

This module gets your machine and your LangSmith workspace ready. You check your packages and keys, register the agent every module traces to, send a first trace, and connect the LangSmith CLI and Remote MCP so a coding agent can read the same data. Last, you can give LangSmith itself a model for Chat, judges and the Playground.

## Setup
Run `uv sync` in a terminal at the repo root, then open this notebook with `uv run jupyter lab` so the kernel uses the project's virtual environment.

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import utils.config  # sets the tracing destination before any LangSmith client exists

packages = ["langsmith", "deepagents", "langchain", "langchain-openai"]
print("Python", sys.version.split()[0], "·", " · ".join(f"{name} {version(name)}" for name in packages))

## 1. Keys and the .env file
Every module reads its keys and its tracing destination from one `.env` file at the repo root. The cell below copies `.env.example` to `.env` only if `.env` doesn't exist yet, so rerunning it never overwrites your keys.

In [ ]:
import shutil

env_file = project_root / ".env"
if env_file.exists():
    print(".env:", env_file, "(already exists)")
else:
    shutil.copy(project_root / ".env.example", env_file)
    print(".env:", env_file, "(created)")

| Key | What it's for | Where to get it |
|---|---|---|
| `LANGSMITH_API_KEY` | This workspace's traces, datasets and rules | **Settings** → **API Keys**, create a **Personal Access Token** |
| `LANGSMITH_GATEWAY_API_KEY` | Model calls through the **LLM Gateway**, the default in `utils/models.py`. Not needed if you switch to your own provider | A key from a workspace with model access; it can be the same as `LANGSMITH_API_KEY` |
| `TAVILY_API_KEY` | The agent's web search | [tavily.com](https://tavily.com) |

Set `LANGSMITH_WORKSPACE_ID` (**Settings** → **General**) if your key spans several workspaces, and `LANGSMITH_ENDPOINT` if your workspace is in the EU, APAC or AWS region. A **Tracing Only** key (`lsv2_tr_…`) can send traces but can't read them, so it can't run these modules.

To use your own provider instead of the Gateway, open `utils/models.py`, uncomment the `MODEL_SPEC` / `API_KEY_ENV` pair for Anthropic, OpenAI, Azure or Bedrock, comment out the Gateway pair, and set that pair's key in `.env` (for example `ANTHROPIC_API_KEY`).

![Create an API key](../images/create_api_key.png)

Fill in `.env`, then run the next cell. It reloads `.env`, prints `set` or `MISSING` for each key, and names where your traces will go.

In [ ]:
import os

from dotenv import load_dotenv

from utils.config import AGENT_ENVIRONMENT, AGENT_NAME, PROJECT_NAME, tracing_mode
from utils.models import API_KEY_ENV

load_dotenv(project_root / ".env", override=True)  # picks up keys you just added

for key in ["LANGSMITH_API_KEY", API_KEY_ENV, "TAVILY_API_KEY"]:  # API_KEY_ENV is the model key utils/models.py uses
    if key is None:
        continue
    print(f"{key}: {'set' if os.getenv(key) else 'MISSING'}")
if os.getenv("LANGSMITH_API_KEY", "").startswith("lsv2_tr_"):
    print("LANGSMITH_API_KEY is a Tracing Only key: create a Personal Access Token instead")

mode = tracing_mode(os.environ)
if mode == "agent":
    print(f"Tracing destination: agent {AGENT_NAME}, environment {AGENT_ENVIRONMENT}")
elif mode == "project":
    print(f"Tracing destination: project {PROJECT_NAME}")
else:
    print("Tracing destination: CONFLICT, keep LANGSMITH_PROJECT or the LANGSMITH_AGENT_* lines, then restart the kernel")

The last line names where every trace goes. With no destination in `.env`, `utils/config.py` traces to `client-research-agent` in Development.

> If `LANGSMITH_PROJECT` is set alongside the agent variables, the SDK has two destinations and traces nothing, so keep exactly one and restart the kernel.

## 2. Register the agent
In an agent-based workspace, every trace belongs to an **agent** and one of its four **environments**: Local, Development, Staging and Production. Registering `client-research-agent` once creates all four; these notebooks trace to Development, and Module 7 deploys to Staging. Every module's traces then land under one agent, so you compare runs by environment instead of juggling project names. The cell below registers the agent, or retrieves it if it already exists, and prints a link to it in **Tracing**.

> **Requires:** an agent-based workspace (LangSmith Cloud). **Without it:** if you're self-hosted or your workspace lists projects instead of agents, set `LANGSMITH_PROJECT` in `.env` (leave the agent lines commented), restart the kernel and run this cell anyway: it creates the project instead.

In [ ]:
from langsmith import Client, ConflictError

from utils.config import TRACING_MODE, environment_project
from utils.notebook import trace_link

client = Client()

if TRACING_MODE == "agent":
    try:
        agent = client.agents.create(id=AGENT_NAME, name="Client research agent", source_type="EXTERNAL")
        print("Agent:", agent.id, "(registered)")
    except ConflictError:
        agent = client.agents.retrieve(AGENT_NAME)
        print("Agent:", agent.id, "(already registered)")
    print("Environments:", ", ".join(env.environment for env in agent.environments))
elif not client.has_project(PROJECT_NAME):
    client.create_project(PROJECT_NAME)
    print("Project:", PROJECT_NAME, "(created)")

dest = environment_project(client)
trace_link(dest.url, "Tracing")

Open the link: **Tracing** shows `client-research-agent` with Development selected. It's empty until the next section.

![Agent in Tracing](../images/TODO-00-agent-development.png)

> Registering needs a key that can create projects; a `403` here means your key lacks that permission.

## 3. Send your first trace
`@traceable` turns any Python function into a trace, with no other code change. The next cell defines `setup_check`, which asks the model for `pong` and runs one web search; the model call becomes a child run because LangChain models trace themselves, and the search result lands in the trace's outputs.

In [ ]:
from langsmith import traceable

from utils.models import make_model
from utils.search import resilient_tavily_search


@traceable(name="setup-check")
def setup_check() -> tuple[str, str]:
    reply = make_model().invoke("Reply with the single word 'pong'.")
    search = resilient_tavily_search("LangSmith observability", max_retries=1)
    return str(reply.text), search



Run it. Expect `Model: pong` and the first line of a search result.

In [ ]:
reply, search = setup_check()

print("Model:", reply)
if search.startswith("[fallback"):
    print("Search: fell back to canned content, check TAVILY_API_KEY")
else:
    print("Search:", search.splitlines()[0])
trace_link(dest.url, "Tracing")

Refresh **Tracing**: the `setup-check` trace has a model child run (`ChatOpenAI` with the default Gateway model) with the `pong` reply, and the search text under **Output**. If search printed the fallback line, fix the Tavily key before Module 2, or the agent's research answers come from canned text.

## 4. The LangSmith CLI
The **LangSmith CLI** reads the same traces, datasets and experiments from a terminal, and a coding agent can call it the same way you do. Install it and log in once in a terminal:

```bash
curl -fsSL https://cli.langsmith.com/install.sh | sh
langsmith auth login
```

Inside this notebook, `!langsmith` uses `LANGSMITH_API_KEY` from `.env`, which takes precedence over your login.

In [ ]:
!langsmith traces list --project-id {dest.project_id} --limit 3

The top row is the `setup-check` trace from section 3.

## 5. LangSmith Remote MCP
The **Remote MCP** server gives any MCP client LangSmith tools for runs, threads, datasets, prompts and experiments at `<your API URL>/mcp`. Coding agents connect over OAuth; the cell below connects with your API key in the `X-Api-Key` header and lists the tools.

In [ ]:
import warnings

from fastmcp.client.transports import StreamableHttpTransport
from langchain_core._api import LangChainBetaWarning

warnings.filterwarnings("ignore", category=LangChainBetaWarning)  # langchain.mcp is in beta
from langchain.mcp import MCPAdapter

transport = StreamableHttpTransport(f"{client.api_url}/mcp", headers={"X-Api-Key": os.environ["LANGSMITH_API_KEY"]})
async with MCPAdapter(transport) as mcp:
    tools = await mcp.list_tools()

print(f"Remote MCP tools ({len(tools)}):", ", ".join(tool.name for tool in tools))

These are the tools a coding agent gets. To add them to Claude Code, run `claude mcp add --transport http -s user langsmith https://api.smith.langchain.com/mcp`, then `/mcp` → **langsmith** to log in; Module 2 asks it about your traces.

> Codex CLI's OAuth login doesn't work with the Remote MCP yet, so from Codex use the LangSmith CLI from section 4.

## 6. Give LangSmith a model
Features that call a model inside LangSmith (Chat, the online judges in Module 4, the Playground) use a **model configuration** in your workspace, not your `.env`. The cell below stores your gateway key as a **workspace secret** and adds a model configuration that calls gpt-5.6-luna through the Gateway with it; it skips either one if it already exists. It only runs once you set `share_gateway_key = True`, because it copies your key into the workspace.

> **Run this cell only if** you're a Workspace Admin **and** your workspace has no model configuration yet. **Otherwise:** skip it. If the Playground lets you pick a model, you're already set; if not, ask an admin to add a provider key under **Settings** → **Workspace Secrets**.

In [ ]:
from utils.config import WORKSPACE_GATEWAY_SECRET, WORKSPACE_MODEL_CONFIG
from utils.models import MODEL_SPEC
from utils.workspace import ensure_model_config, ensure_workspace_secret

share_gateway_key = False  # admins with no model configuration yet: set True to store LANGSMITH_GATEWAY_API_KEY here

if share_gateway_key:
    secret_status = ensure_workspace_secret(client, WORKSPACE_GATEWAY_SECRET, os.environ["LANGSMITH_GATEWAY_API_KEY"])
    print(f"Workspace secret {WORKSPACE_GATEWAY_SECRET}: {secret_status}")
    config_status = ensure_model_config(client, WORKSPACE_MODEL_CONFIG, MODEL_SPEC, WORKSPACE_GATEWAY_SECRET)
    print(f"Model configuration {WORKSPACE_MODEL_CONFIG}: {config_status}")
else:
    print("Skipped: set share_gateway_key = True to give LangSmith a model")

Open **Settings** → **Workspace Secrets**: `GATEWAY_MODEL_API_KEY` is listed with its value hidden. Under **Settings** → **Model configurations**, `client-research-model` is available to the Playground, evaluators and Chat.

![Model configuration](../images/TODO-00-model-configuration.png)

## Recap
| Step | Where it lives |
|---|---|
| `.env` | Repo root, read by `utils/config.py` |
| Agent and environments | **Tracing** agent switcher · `client.agents.create` |
| First trace | **Tracing** → Development · `@traceable` |
| CLI | `langsmith traces list` |
| Remote MCP | `<API URL>/mcp` · `MCPAdapter` |
| Model for LangSmith | **Settings** → **Workspace Secrets** and model configurations · `utils/workspace.py` |

**Next:** Module 1 — Build a deep agent (optional), or Module 2 — Tracing.

**Docs:** [Create an agent](https://docs.langchain.com/langsmith/create-an-agent) · [Log traces to an agent](https://docs.langchain.com/langsmith/log-traces-to-agent) · [API keys](https://docs.langchain.com/langsmith/create-account-api-key) · [LangSmith CLI](https://docs.langchain.com/langsmith/langsmith-cli) · [Remote MCP](https://docs.langchain.com/langsmith/langsmith-remote-mcp) · [LLM Gateway](https://docs.langchain.com/langsmith/llm-gateway-direct-model-access)